<a href="https://colab.research.google.com/github/malyadri6761/RAG/blob/master/rag_02.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

---

**Install Libararies**

----

In [1]:
!pip install -q sentence-transformers faiss-cpu openai


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 72.6 MB/s eta 0:00:00


---

**Sample Documents**

---

In [2]:
docs = [
    {
        "text": """
        Artificial Intelligence is a field of computer science that focuses
        on creating systems capable of performing tasks that normally require
        human intelligence.

        Machine learning is a subset of artificial intelligence. It allows
        computers to learn patterns from data and make predictions or decisions
        without being explicitly programmed for every task.

        Deep learning is a subset of machine learning that uses neural networks
        with multiple layers to learn complex patterns from large amounts of data.
        """,
        "source": "ai.txt"
    },

    {
        "text": """
        Supervised learning is a machine learning approach where a model learns
        from labeled training data.

        Classification is a supervised learning task where the goal is to predict
        a discrete class or category.

        Regression is a supervised learning task where the goal is to predict
        a continuous numerical value.

        Unsupervised learning works with data that does not have labeled outputs.
        Clustering is one example of unsupervised learning.
        """,
        "source": "ml.txt"
    }
]

In [3]:
print(docs)

[{'text': '\n        Artificial Intelligence is a field of computer science that focuses\n        on creating systems capable of performing tasks that normally require\n        human intelligence.\n\n        Machine learning is a subset of artificial intelligence. It allows\n        computers to learn patterns from data and make predictions or decisions\n        without being explicitly programmed for every task.\n\n        Deep learning is a subset of machine learning that uses neural networks\n        with multiple layers to learn complex patterns from large amounts of data.\n        ', 'source': 'ai.txt'}, {'text': '\n        Supervised learning is a machine learning approach where a model learns\n        from labeled training data.\n\n        Classification is a supervised learning task where the goal is to predict\n        a discrete class or category.\n\n        Regression is a supervised learning task where the goal is to predict\n        a continuous numerical value.\n\n       

---

**3. Create Chunks**

---

In [12]:
def create_chunks(text,chunk_size=10):
  words = text.split()
  chunks = []

  for i in range(0,len(words),chunk_size):
    chunk = " ".join(words[i:i+chunk_size])
    chunks.append(chunk)

  return chunks

---

**4. Create all_chunks**

---

In [14]:
all_chunks = []

for document in docs:
  text = document['text']
  chunks = create_chunks(text)

  for chunk in chunks:
    all_chunks.append({
        'text':chunk,
        'source':document['source']
    })




In [17]:
for i,chunk in enumerate(all_chunks):
  print(f'chunk : {i}')
  print("Source : ",chunk["source"])
  print("Text : ",chunk["text"])
  print("-"*30)



chunk : 0
Source :  ai.txt
Text :  Artificial Intelligence is a field of computer science that focuses
------------------------------
chunk : 1
Source :  ai.txt
Text :  on creating systems capable of performing tasks that normally require
------------------------------
chunk : 2
Source :  ai.txt
Text :  human intelligence. Machine learning is a subset of artificial intelligence.
------------------------------
chunk : 3
Source :  ai.txt
Text :  It allows computers to learn patterns from data and make
------------------------------
chunk : 4
Source :  ai.txt
Text :  predictions or decisions without being explicitly programmed for every task.
------------------------------
chunk : 5
Source :  ai.txt
Text :  Deep learning is a subset of machine learning that uses
------------------------------
chunk : 6
Source :  ai.txt
Text :  neural networks with multiple layers to learn complex patterns from
------------------------------
chunk : 7
Source :  ai.txt
Text :  large amounts of data.
-------

---

**5. Load embedding model**

---

In [19]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

---

**6. Extract Chunk text**

---

In [20]:
texts = [chunk["text"] for chunk in all_chunks]
print(texts)

['Artificial Intelligence is a field of computer science that focuses', 'on creating systems capable of performing tasks that normally require', 'human intelligence. Machine learning is a subset of artificial intelligence.', 'It allows computers to learn patterns from data and make', 'predictions or decisions without being explicitly programmed for every task.', 'Deep learning is a subset of machine learning that uses', 'neural networks with multiple layers to learn complex patterns from', 'large amounts of data.', 'Supervised learning is a machine learning approach where a model', 'learns from labeled training data. Classification is a supervised learning', 'task where the goal is to predict a discrete class', 'or category. Regression is a supervised learning task where the', 'goal is to predict a continuous numerical value. Unsupervised learning', 'works with data that does not have labeled outputs. Clustering', 'is one example of unsupervised learning.']


---

**7. Generate Embedddings**

---

In [21]:
embeddings = model.encode(texts)
print(embeddings.shape)

(15, 384)


---

**8. Create FAISS vector database**

---

In [23]:
import faiss
dimension = embeddings.shape[1]

index = faiss.IndexFlatL2(dimension)
index.add(embeddings)

print("No.of Vectors : ",index.ntotal)

No.of Vectors :  15


---

**9. Create a user Query**

---

In [28]:
query = "What is supervised learning?"

query_embedding = model.encode([query])
print(query_embedding.shape)

(1, 384)


---

**10. Search FAISS**

---

In [29]:
distance,indices = index.search(
    query_embedding,
    k=3
)

print("Indices : ",indices)
print("Distance : ",distance)

Indices :  [[ 8  9 14]]
Distance :  [[0.29673257 0.4185029  0.9468646 ]]


---

**11. Retrieve actual text**

---

```
  Query
    ↓
  Query Embedding
    ↓
  FAISS
    ↓
  Top 3 chunks


In [30]:
for i in indices[0]:

    print("Source:", all_chunks[i]["source"])
    print("Text:", all_chunks[i]["text"])
    print("-" * 50)

Source: ml.txt
Text: Supervised learning is a machine learning approach where a model
--------------------------------------------------
Source: ml.txt
Text: learns from labeled training data. Classification is a supervised learning
--------------------------------------------------
Source: ml.txt
Text: is one example of unsupervised learning.
--------------------------------------------------


---

**12. Build the context**

---

In [31]:
context = "\n\n".join(
    all_chunks[i]["text"]
    for i in indices[0]
)

In [32]:
print(context)

Supervised learning is a machine learning approach where a model

learns from labeled training data. Classification is a supervised learning

is one example of unsupervised learning.


---

**13. Create the RAG prompt**

---

```
                Context
                 +
               Query
                 ↓
               Prompt
                 ↓
                LLM
```

In [33]:
prompt = f"""
Answer the question using only the provided context.

Context:
{context}

Question:
{query}

Answer:
"""

In [39]:
import os
from google.colab import userdata

os.environ['GROQ_API_KEY'] = userdata.get('GROQ_API_KEY')

---

**14. Groq Client**

---

In [52]:
# !pip install groq
from groq import Groq
model = Groq(api_key = userdata.get("GROQ_API_KEY"))

In [53]:
response = model.chat.completions.create(
    model="openai/gpt-oss-120b",
    messages=[
        {
            "role": "user",
            "content": prompt
        }
    ],
    temperature=0
)

answer = response.choices[0].message.content

print("Answer:")
print(answer)

Answer:
Supervised learning is a machine learning approach where a model learns from labeled training data.
